# CS 3110/5110: Data Privacy
## Homework 8

In [11]:
# Load the data and libraries
import pandas as pd
import numpy as np
import random
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def laplace_mech_vec(vec, sensitivity, epsilon):
    return [v + np.random.laplace(loc=0, scale=sensitivity / epsilon) for v in vec]

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)
            for v in vec]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

In [12]:
# preserves epsilon-differential privacy
def above_threshold(queries, df, T, epsilon):
    T_hat = T + np.random.laplace(loc=0, scale = 2/epsilon)
    
    for idx, q in enumerate(queries):
        nu_i = np.random.laplace(loc=0, scale = 4/epsilon)
        if q(df) + nu_i >= T_hat:
            return idx
    return -1 # the index of the last element

## Question 1 (20 points)

Implement a function `above_10000` that releases the **value** of the first query in a sequence of queries whose value is above 10000. Your function should have a **total** privacy cost equal to the privacy parameter $\epsilon$ passed in when it is called.

In [13]:
def make_query(status):
    def q(df):
        return len(df[df['Marital Status'] == status])
    return q

def above_10000(queries, epsilon):
    T = 1000
    idx = above_threshold(queries, adult, T=T, epsilon=epsilon/2)
    qeuery_value = queries[idx](adult)
    noisy_value = laplace_mech(qeuery_value, sensitivity=1, epsilon=epsilon/2)
    return noisy_value
    

queries = [make_query(status) for status in adult['Marital Status'].unique()]
print(f"above_10000 #1: {above_10000(queries, 100)}")
print(f"above_10000 #2: {above_10000(queries, 1)}")
print(f"above_10000 #3: {above_10000(queries, .01)}")

above_10000 #1: 10682.978421285266
above_10000 #2: 10681.813558742066
above_10000 #3: 10642.930486994608


In [14]:
# TEST CASE

results = [above_10000(queries, 1.0) for _ in range(100)]
print(np.mean(results))
assert np.mean(results) > 9900
assert np.mean(results) < 11000

10683.2303647125


## Question 2 (10 points)
In 2-3 sentences, argue informally (via the definition of the sparse vector technique, post-processing, and sequential composition), that your implementation of `above_10000` has a total privacy cost of $\epsilon$.

### Sensitivity: 
The sensivity of laplace_mech is 1 because the queries used are counting queries, so adding or removing a person changes the neighboring dataset by 1.

### Composition
above_threshold uses half of the $\epsilon$ budget by the sparse vector technique. laplace_mech uses the other half of the $\epsilon$ budget to add noise to the query_value. By sequential composition, the total privacy cost is $\epsilon$/2 + $\epsilon$/2 = $\epsilon$.

### Post-Processing
The index returned by above_threshold satisfies DP by the sparse vector technique, the qeuery_value also satisfies DP by using the laplace mechanism, and so the return noisy value. Nothing can be done to the output of the function that will leak any privacy, so the query as whole satisfies DP.


## Question 3 (20 points)

Implement a function `bounded_all_above_10000` that releases the **value** of **$c$ queries** in a sequence of queries whose value is above 10000 (where $c$ is an analyst-provided parameter limiting the number of returned results). Your function should have a **total privacy cost** bounded by its parameter $\epsilon$.

In [15]:
def bounded_all_above_10000(queries, c, epsilon):
    T = 1000
    queries_temp_in = queries
    noisy_vals = []
    epsilon_c = epsilon/c
    for _ in range(c):
        if not queries_temp_in:
            break
        
        idx = above_threshold(queries_temp_in, adult, T=T, epsilon=epsilon_c/2)

        if idx == -1:
            break

        true_val = queries_temp_in[idx](adult)
        noisy_val = laplace_mech(true_val, sensitivity=1, epsilon=epsilon_c/2)
        noisy_vals.append(noisy_val)

        queries_temp_in = queries_temp_in[idx + 1:]

    
    return noisy_vals

queries = [make_query(status) for status in adult['Marital Status'].unique()]
print(f"bounded_all_above_10000 #1: {bounded_all_above_10000(queries, 3, 100)}")
print(f"bounded_all_above_10000 #2: {bounded_all_above_10000(queries, 3, 1)}")
print(f"bounded_all_above_10000 #3: {bounded_all_above_10000(queries, 3, .01)}")

bounded_all_above_10000 #1: [10683.060947923193, 14976.002526079834, 4443.088661583383]
bounded_all_above_10000 #2: [10686.238519812341, 14971.15796456847, 4441.982624799649]
bounded_all_above_10000 #3: [11085.38545723608, 15010.065466795519, 3946.992810626593]


In [16]:
# TEST CASE

results = [bounded_all_above_10000(queries, 2, 1.0) for _ in range(100)]
results_1 = [r[0] for r in results]
results_2 = [r[1] for r in results]

assert np.mean(results_1) > 9900
assert np.mean(results_1) < 11000
assert np.mean(results_2) > 14000
assert np.mean(results_2) < 15500

## Question 4 (10 points)

In 2-3 sentences, argue informally that your implementation of `bounded_all_above_10000` has privacy cost bounded by $\epsilon$.

### Sensitivity: 
The sensivity of laplace_mech is 1 because the queries used are counting queries, so adding or removing a person changes the neighboring dataset by 1. 

### Composition
$\epsilon$ is divided by c, so each iteration of the loop uses $\epsilon$/c of the budget by sequential composition. Inside each iteration of the loop, above threshold uses uses half of the $\epsilon$/c budget by the sparse vector technique and laplace_mech uses the other half of the $\epsilon$/c budget to add noise to the query_value. By sequential composition, the total $\epsilon$ cost is 
c($\epsilon$/2c + $\epsilon$/2c) = $\epsilon$.

### Post-Processing
Both the indices returned by above_threshold and the noisy values using the aforementioned indices satisfy DP, so appending them to a list and returning said list are valid post-processing steps because they only manipulate the noisy outputs. Nothing can be done to the output of the function that will leak any privacy, so the query as whole satisfies DP.


## Question 5 (30 points)

Implement a function `mean_age` that computes the mean age of participants in the `adult_data` dataset. Your function should have a **total** privacy cost of $\epsilon$. It should work as follows:

1. Compute an upper clipping parameter based on the data using a differentially private algorithm
2. Clip the data using the clipping parameter
3. Use `laplace_mech` to release a differentially private mean of the clipped data

*Hint*: Use the sparse vector technique (`above_threshold`) to compute the clipping parameter. Consider using a sequence of queries that looks like `df.clip(lower=0, upper=b).sum() - df.clip(lower=0, upper=b+1).sum()`.

*Hint*: Be careful of sensitivities and set the scale of the noise accordingly!

In [ ]:
bs = list(range(0, 200, 10))
df = adult['Age']
b_lower = 0

def make_query(b):
    def q(df):
        return df["Age"].clip(lower=0, upper=b).sum() - df["Age"].clip(lower=0, upper=b+1).sum()
    return q

def mean_age(epsilon):
    queries = [make_query(b) for b in range(1, 1000)]
    b = above_threshold(queries, adult,T=0, epsilon=epsilon/3)
    
    sum_clipped = adult["Age"].clip(lower=0, upper=b).sum()
    noisy_sum = laplace_mech(sum_clipped, sensitivity=b, epsilon=epsilon/3)
    true_count = len(adult)
    noisy_count = laplace_mech(true_count, sensitivity=1, epsilon=epsilon/3)
    noisy_mean = noisy_sum / noisy_count
    return noisy_mean
    
for epsilon in [0.001, 0.01, 0.1, 0.5, 1, 10]:
    print(f"epsilon: {epsilon}, mean age: {mean_age(epsilon)}")

epsilon: 0.001, mean age: 9.21200353332122
epsilon: 0.01, mean age: 37.49504761494444
epsilon: 0.1, mean age: 38.65144509055952
epsilon: 0.5, mean age: 38.580548358403945
epsilon: 1, mean age: 38.58928868093675
epsilon: 10, mean age: 38.58091392701618


In [20]:
# TEST CASE
results = [mean_age(1.0) for _ in range(100)]
assert np.mean(results) > 38
assert np.mean(results) < 39

## Question 6 (10 points)

In 3-5 sentences, describe your approach to implementing `mean_age` and argue informally that your implementation has privacy cost bounded by $\epsilon$.

### Approach
I found the clipping parameter by finding when the sum of ages stopped increasing.

I found the noisy mean by dividing the noisy sum by the noisy count.

### Sensitivity: 
The sensivity of the laplace_mech used for the noisy sum is the clipping parameter b because that's the most a neighboring dataset could differ by. The sensitivity for the noisy count is 1 because it's a counting query.

### Composition
$\epsilon$/3 is used to find the clipping paramter, $\epsilon$/3 is used to find the noisy sum, $\epsilon$/3 is used to find the noisy count. By sequential composition, $\epsilon$/3 + $\epsilon$/3 + $\epsilon$/3 = $\epsilon$

### Post-Processing
Both the noisy sum and the noisy count satisfy DP, so using them to compute a noisy sum also satisfies DP. Nothing can be done to the output of the function that will leak any privacy, so the query as whole satisfies DP.


## Question 7 (20 points)

Write an algorithm to compute the (approximate) maximum of a given column of the adult dataset. Your solution should satisfy `epsilon` differential privacy.

*Hint*: you might want to use AboveThreshold; you might need to test several options for building your queries and for the threshold

In [23]:
def make_query(b, column):
    def q(df):
        # Number of people greater than b
        count = len(df[df[column] > b])

        # above_threshold finds the first query that is above the treshold, so to find the first query below the threshold multiply the count by -1
        # Values increase until they reach 0
        count *= -1
        return count
    return q
def max_of_col(column, epsilon):
    r = [2**(i/4) for i in range(22)] 

    queries = [make_query(b, column) for b in r]

    # Stops at the first query with 0 people
    idx = above_threshold(queries=queries, df=adult, T=0, epsilon=epsilon)
    return r[idx]




max_of_col('Age', 1.0)

38.05462768008707

In [24]:
# TEST CASE
cols = ['Age', 'Capital Gain', 'fnlwgt']

for c in cols:
    true_val = adult[c].max()
    trials = [max_of_col(c, 10.0) for _ in range(20)]
    errors = [pct_error(true_val, t) for t in trials]
    print('Median error for column ' + c + ':', np.median(errors))
    assert np.median(errors) > 0
    assert np.median(errors) < 100

Median error for column Age: 57.717080355458805
Median error for column Capital Gain: 99.96194499176983
Median error for column fnlwgt: 99.99743688963936


## Question 8 (10 points)

In 3-5 sentences, describe your approach to implementing `max_of_col` and argue that your approach has total privacy cost bounded by $\epsilon$.

### Approach
To find the maximum value using above threshold, I built a query that checks how many people have a value greater than b. (The 1st value .of b such that nobody has a value greater than b is the greatest value). Since above threshold checks for values that exceed the threshold and I wanted to find the first value below the threshold, I multiplied the count by -1 (values now increase to 0 instead of decreasing to 0).

### Sensitivity: 


### Composition
$\epsilon$

### Post-Processing
r[idx] does not depend on the data, so it does is a valid 
